# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen


El objetivo es entrenar un modelo de Random Forest sobre TF-IDF con búsqueda de hiperparámetros en dos fases (RandomizedSearch amplio y despues uno fino), incorporando feature selection con SelectKBest sobre chi² para reducir ruido y controlar overfitting. Además se exploran dos nuevos hiperparámetros regularizadores (min_samples_split y max_leaf_nodes) no utilizados en entregas anteriores. Los resultados se comparan contra el baseline de Naive Bayes (~0.66 F1-macro en Kaggle).


## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 90.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, learning_curve
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import f1_score


In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)


## Carga de datos y preprocesamiento


In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)


In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")


Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N5: Random Forest + TF-IDF con SelectKBest y dos fases de RandomizedSearch


Pipeline de tres etapas: TF-IDF → SelectKBest(chi²) → RandomForestClassifier.
SelectKBest filtra features ruidosas antes del clasificador, lo que reduce el overfitting y acelera el entrenamiento.

Se exploran **nueve** hiperparámetros, incluyendo dos que nunca se usaron en entregas anteriores:
- `rf__min_samples_split`: mínimo de muestras para dividir un nodo (nuevo)
- `rf__max_leaf_nodes`: cota al tamaño de cada árbol (nuevo)

También se agrega `select__k` para controlar cuántas features retiene SelectKBest.

La búsqueda se realiza en **dos fases** para maximizar el F1 sin reventar el tiempo de Colab:

1. **RandomizedSearchCV amplio** (20 iter, 5-fold): explora combinaciones de todos los parámetros.
2. **RandomizedSearchCV refinado** (12 iter, 3-fold): búsqueda más acotada alrededor del mejor punto
   de la Fase 1 para ajustar el vecindario óptimo.

Hiperparámetros explorados (Fase 1):
| Parámetro | Valores |
|-----------|---------|
| tfidf__ngram_range | (1,1), (1,2) |
| tfidf__min_df | 2, 3, 5, 10 |
| select__k | 5K, 10K, 20K, 50K |
| rf__n_estimators | 300, 500, 700 |
| rf__max_depth | 30, 50, 80, None |
| rf__min_samples_split | 2, 5, 10, 20 |
| rf__min_samples_leaf | 1, 2, 4, 8 |
| rf__max_leaf_nodes | None, 500, 1000, 5000 |
| rf__max_features | sqrt, log2 |



### Pipeline


In [ ]:
pipe = Pipeline([
    ("tfidf",  make_tfidf()),
    ("select", SelectKBest(chi2)),
    ("rf",     RandomForestClassifier(
                   class_weight="balanced",
                   random_state=SEED,
                   n_jobs=2
               ))
])


### Fase 1: RandomizedSearchCV amplio (20 iter, 5-fold)

Se explora el espacio completo de 9 hiperparámetros con 20 combinaciones aleatorias.


In [ ]:
param_dist = {
    "tfidf__ngram_range":      [(1, 1), (1, 2)],
    "tfidf__min_df":           [2, 3, 5, 10],
    "select__k":               [5000, 10000, 20000, 50000],
    "rf__n_estimators":        [300, 500, 700],
    "rf__max_depth":           [30, 50, 80, None],
    "rf__min_samples_split":   [2, 5, 10, 20],
    "rf__min_samples_leaf":    [1, 2, 4, 8],
    "rf__max_leaf_nodes":      [None, 500, 1000, 5000],
    "rf__max_features":        ["sqrt", "log2"],
}

rs = RandomizedSearchCV(
    pipe, param_dist,
    n_iter=20, cv=5,
    scoring="f1_macro",
    n_jobs=2,
    random_state=SEED,
    verbose=1
)

print("Fase 1: RandomizedSearchCV amplio (20 iter, 5-fold)")
rs.fit(X_train, y_train)

print(f"\nMejores hiperparámetros (RS amplio): {rs.best_params_}")
print(f"Mejor F1-macro en CV (RS amplio):     {rs.best_score_:.4f}")

y_pred_rs = rs.best_estimator_.predict(X_val)
evaluate("rf_tfidf_select_rs_v5", y_val, y_pred_rs, hyperparams=rs.best_params_)


Fase 1: RandomizedSearchCV amplio (20 iter, 5-fold)
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=50000 is greater than n_features=41409. All the features will be returned.
  warnings.warn(



Mejores hiperparámetros (RS amplio): {'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 3, 'select__k': 50000, 'rf__n_estimators': 700, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 4, 'rf__max_leaf_nodes': None, 'rf__max_features': 'sqrt', 'rf__max_depth': None}
Mejor F1-macro en CV (RS amplio):     0.6423

=== rf_tfidf_select_rs_v5 ===
Hiperparámetros: {'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 3, 'select__k': 50000, 'rf__n_estimators': 700, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 4, 'rf__max_leaf_nodes': None, 'rf__max_features': 'sqrt', 'rf__max_depth': None}

F1-macro:  0.6491
Precision: 0.6492
Recall:    0.6493
Accuracy:  0.6932

              precision    recall  f1-score   support

    negativa     0.7462    0.7544    0.7503      4080
      neutra     0.4078    0.4294    0.4183      2040
    positiva     0.7937    0.7640    0.7786      4080

    accuracy                         0.6932     10200
   macro avg     0.6492    0.6493    0.6491     10200
weighted 

### Fase 2: RandomizedSearchCV refinado (12 iter, 3-fold)

Una vez conocido el vecindario del óptimo (Fase 1), se ajusta fino con rangos más angostos alrededor del mejor punto. Solo 12 iteraciones y 3-fold para ser rápidos.



In [ ]:
def narrow_range(val, step, min_val=None, max_val=None):
    # Genera 2-3 valores cercanos a val con paso step, respetando limites
    if val is None:
        return None
    vals = [val]
    if min_val is None or val - step >= min_val:
        vals.append(val - step)
    if max_val is None or val + step <= max_val:
        vals.append(val + step)
    return sorted(set(vals))

best = rs.best_params_
param_refined = {}
for k in ["tfidf__ngram_range", "rf__max_leaf_nodes", "rf__max_features"]:
    param_refined[k] = [best.get(k)]

mdf = best.get("tfidf__min_df", 5)
param_refined["tfidf__min_df"] = narrow_range(mdf, 1, min_val=1)

sk = best.get("select__k", 20000)
param_refined["select__k"] = narrow_range(sk, 5000, min_val=1000)

ne = best.get("rf__n_estimators", 500)
param_refined["rf__n_estimators"] = narrow_range(ne, 100, min_val=100)

md = best.get("rf__max_depth")
if md is None:
    param_refined["rf__max_depth"] = [None, 80, 100]
else:
    param_refined["rf__max_depth"] = narrow_range(md, 20, min_val=10)

mss = best.get("rf__min_samples_split", 5)
param_refined["rf__min_samples_split"] = narrow_range(mss, 2, min_val=2)

msl = best.get("rf__min_samples_leaf", 2)
param_refined["rf__min_samples_leaf"] = narrow_range(msl, 1, min_val=1)

# Remover entradas None (ya se manejan dentro de narrow_range)
param_refined = {k: v for k, v in param_refined.items() if v is not None}

print("Rangos de la Fase 2 (refinado):")
for k, v in param_refined.items():
    print(f"  {k}: {v}")

print("\nFase 2: RandomizedSearchCV refinado (12 iter, 3-fold)")
rs2 = RandomizedSearchCV(
    pipe, param_refined,
    n_iter=12, cv=3,
    scoring="f1_macro",
    n_jobs=2,
    random_state=SEED,
    verbose=1
)
rs2.fit(X_train, y_train)

print(f"\nMejores hiperparámetros (RS refinado): {rs2.best_params_}")
print(f"Mejor F1-macro en CV (RS refinado):     {rs2.best_score_:.4f}")

y_pred_rs2 = rs2.best_estimator_.predict(X_val)
evaluate("rf_tfidf_select_rs2_v5", y_val, y_pred_rs2, hyperparams=rs2.best_params_)

# Seleccionar el mejor entre ambas fases
if rs2.best_score_ > rs.best_score_:
    best_estimator = rs2.best_estimator_
    best_params_final = rs2.best_params_
    best_cv_score = rs2.best_score_
    source = "RS refinado (Fase 2)"
else:
    best_estimator = rs.best_estimator_
    best_params_final = rs.best_params_
    best_cv_score = rs.best_score_
    source = "RS amplio (Fase 1)"

print(f"\nModelo final seleccionado: {source}")
print(f"F1-macro CV: {best_cv_score:.4f}")


Rangos de la Fase 2 (refinado):
  tfidf__ngram_range: [(1, 2)]
  rf__max_leaf_nodes: [None]
  rf__max_features: ['sqrt']
  tfidf__min_df: [2, 3, 4]
  select__k: [45000, 50000, 55000]
  rf__n_estimators: [600, 700, 800]
  rf__max_depth: [None, 80, 100]
  rf__min_samples_split: [3, 5, 7]
  rf__min_samples_leaf: [3, 4, 5]

Fase 2: RandomizedSearchCV refinado (12 iter, 3-fold)
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Mejores hiperparámetros (RS refinado): {'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'select__k': 45000, 'rf__n_estimators': 800, 'rf__min_samples_split': 7, 'rf__min_samples_leaf': 3, 'rf__max_leaf_nodes': None, 'rf__max_features': 'sqrt', 'rf__max_depth': None}
Mejor F1-macro en CV (RS refinado):     0.6406

=== rf_tfidf_select_rs2_v5 ===
Hiperparámetros: {'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'select__k': 45000, 'rf__n_estimators': 800, 'rf__min_samples_split': 7, 'rf__min_samples_leaf': 3, 'rf__max_leaf_nodes': None, 'rf__max_features'

### Modelo final: reentrenar, guardar y generar submission

Se reentrena el mejor pipeline sobre *todo* el dataset (train + val) y se genera
el archivo CSV para subir a Kaggle.


In [ ]:
print("Reentrenando sobre dataset completo (train + val)...")
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])
best_estimator.fit(X_full, y_full)

save_model(best_estimator, "rf_tfidf_select_v5")
make_submission(
    best_estimator,
    pd.DataFrame({"id": test["id"], "text": X_test}),
    "submission_rf_tfidf_select_v5.csv"
);


Reentrenando sobre dataset completo (train + val)...
Modelo guardado → models/rf_tfidf_select_v5.joblib
Predicción guardada → submissions/submission_rf_tfidf_select_v5.csv  (8500 predicciones)
Distribución: clase 0: 40.6%, clase 1: 20.4%, clase 2: 39.0%
